In [ ]:
import pandas as pd

df_reddit = pd.read_csv("../../data/raw/reddit_depression_dataset.csv")
df_reddit.head()

In [ ]:
df_reddit = df_reddit[df_reddit["label"].isin([1])]

In [ ]:
df_reddit.subreddit.value_counts()

In [ ]:
import sys

sys.path.insert(0, "../../")  # go up from notebooks/stan/ to project root

from src.data_cleaning.data import clean_data

df_reddit_cleaned = clean_data(df_reddit)

In [ ]:
import re, html, string


def clean_text(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"r/ ?\w+", "", text)
    text = text.replace("...", "three_dots")
    punctuation_to_remove = string.punctuation.replace("!", "").replace("?", "").replace("'", "")
    text = re.sub(f"[{re.escape(punctuation_to_remove)}]", "", text)
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df_reddit_cleaned["title"] = df_reddit_cleaned["title"].apply(clean_text)

In [ ]:
df_emotion_unsplit = pd.read_csv("../../data/raw/emotion_unsplit.csv")
df_emotion_unsplit.head()

In [ ]:
df_emotion_unsplit.label.value_counts()

In [ ]:
df_emotion_positive = df_emotion_unsplit[~df_emotion_unsplit["label"].isin([0, 3, 4])]

In [ ]:
df_emotion_positive.loc[:, "label"] = 0

df_reddit_cleaned.rename(columns={"title": "text"}, inplace=True)

df = pd.concat([df_reddit_cleaned[["text", "label"]], df_emotion_positive[["text", "label"]]], ignore_index=True)

In [ ]:
df.label.value_counts()

In [ ]:
import xml.etree.ElementTree as ET
import glob

records = []
file_num = 0
for filepath in glob.glob("/Users/grinchenko.stanislav/Downloads/t1-depression-symptom-ranking/erisk25-t1-dataset/s_*.trec"):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f"<root>{f.read()}</root>"
    root = ET.fromstring(content)
    for doc in root.findall("DOC"):
        docno = doc.find("DOCNO").text.strip()
        text = (doc.find("TEXT").text or "").strip()
        records.append({"doc_id": docno, "text": text})
    print(f"Processed file {file_num}: {filepath} with {len(records)} records so far.")
    file_num += 1

df_trec = pd.DataFrame(records)
print(df_trec.shape)

In [ ]:
df_pos = df[df["label"] == 1].sample(n=190000, random_state=42)
df_neg = df[df["label"] == 0]

df_balanced = pd.concat([df_pos, df_neg], ignore_index=True)

In [ ]:
# Cell A — Clean emotion text (was skipped earlier)
df_emotion_positive = df_emotion_positive.copy()
df_emotion_positive["text"] = df_emotion_positive["text"].apply(clean_text)

In [ ]:
# Cell B — Integrate eRisk25 as label=1
df_trec["text"] = df_trec["text"].apply(clean_text)
df_trec["label"] = 1.0
df_erisk = df_trec[["text", "label"]].dropna()
print(f"eRisk25 records: {len(df_erisk)}")

In [ ]:
# Cell C — Rebuild combined df and re-balance (1:1 ratio), then shuffle
df = pd.concat([df_reddit_cleaned[["text", "label"]], df_erisk, df_emotion_positive[["text", "label"]]], ignore_index=True)
print("Before balancing:", df["label"].value_counts().to_dict())

n_neg = int((df["label"] == 0).sum())
df_pos = df[df["label"] == 1].sample(n=n_neg, random_state=42)
df_neg = df[df["label"] == 0]
df_balanced = pd.concat([df_pos, df_neg], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print("After balancing:", df_balanced["label"].value_counts().to_dict())

In [ ]:
# Cell D — Sanity check
print(df_balanced.shape)
print(df_balanced["label"].value_counts())
df_balanced.sample(5)

In [ ]:
# Cell E — Train / val / test split (70 / 15 / 15, stratified)
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df_balanced, test_size=0.30, stratify=df_balanced["label"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=42)

print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")

In [ ]:
# Cell F — Save outputs
import os

out_dir = "../../data/processed"
os.makedirs(out_dir, exist_ok=True)

df_balanced.to_csv(f"{out_dir}/final_dataset_v2.csv", index=False)
train.to_csv(f"{out_dir}/train_v2.csv", index=False)
val.to_csv(f"{out_dir}/val_v2.csv", index=False)
test.to_csv(f"{out_dir}/test_v2.csv", index=False)

print("Saved to data/processed/:")
print(f"  final_dataset_v2.csv  → {len(df_balanced):,} rows")
print(f"  train_v2.csv          → {len(train):,} rows")
print(f"  val_v2.csv            → {len(val):,} rows")
print(f"  test_v2.csv           → {len(test):,} rows")